# Cuisine inference and pantry completion

Train the lesson’s linear SVC, export it with sklearn-onnx, and verify ONNX probabilities against Scikit-learn. The new assignment adds a pantry-completion recommender: estimate cuisine from the selected ingredients, then suggest frequent missing ingredients from **training recipes only**. This is an ingredient suggestion demo, not a guarantee of a recipe or dietary suitability.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

raw = pd.read_csv(Path("../../../4-Classification/data/cuisines.csv"))
X = raw.drop(columns=["cuisine", "Unnamed: 0", "rice", "garlic", "ginger"]).astype(float)
y = raw.cuisine
# Group identical feature vectors, even when their labels disagree.
groups = pd.util.hash_pandas_object(X, index=False)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
assert X.shape[1] == 380 and X.isin([0, 1]).all().all()
print(f"Rows: {len(X)}; train: {len(X_train)}; test: {len(X_test)}; features: {X.shape[1]}")


Rows: 2448; train: 1960; test: 488; features: 380


In [2]:
import json
import onnx
import onnxruntime as ort
from sklearn.svm import SVC
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

model = SVC(kernel="linear", C=10, probability=True, class_weight="balanced", random_state=0)
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(classification_report(y_test, prediction, zero_division=0))
print("Test accuracy:", accuracy_score(y_test, prediction))
print("Test macro F1:", f1_score(y_test, prediction, average="macro"))
index_names = list(pd.read_csv("../../../4-Classification/data/ingredient_indexes.csv").columns)
assert list(X.columns) == index_names
print("All 380 feature names match the lesson ingredient index.")

              precision    recall  f1-score   support

     chinese       0.61      0.70      0.66        88
      indian       0.91      0.93      0.92       124
    japanese       0.61      0.57      0.59        61
      korean       0.88      0.80      0.84       152
        thai       0.78      0.81      0.80        63

    accuracy                           0.79       488
   macro avg       0.76      0.76      0.76       488
weighted avg       0.79      0.79      0.79       488

Test accuracy: 0.7889344262295082
Test macro F1: 0.7607787933534232
All 380 feature names match the lesson ingredient index.


In [3]:
converted = convert_sklearn(model, initial_types=[("ingredients", FloatTensorType([None, X.shape[1]]))], options={id(model): {"zipmap": False}}, target_opset=17)
onnx.checker.check_model(converted)
Path("model.onnx").write_bytes(converted.SerializeToString())
options = ort.SessionOptions()
options.log_severity_level = 3
session = ort.InferenceSession("model.onnx", sess_options=options, providers=["CPUExecutionProvider"])
onnx_labels, onnx_probabilities = session.run(None, {"ingredients": X_test.to_numpy(dtype=np.float32)})
np.testing.assert_array_equal(onnx_labels, prediction)
np.testing.assert_allclose(onnx_probabilities, model.predict_proba(X_test), atol=2e-5, rtol=2e-5)
print("All held-out labels and probabilities match after ONNX conversion.")
print("ONNX graph:", [(node.domain, node.op_type) for node in converted.graph.node])
print("Inputs:", [(item.name, item.shape, item.type) for item in session.get_inputs()])
print("Outputs:", [(item.name, item.shape, item.type) for item in session.get_outputs()])

All held-out labels and probabilities match after ONNX conversion.
ONNX graph: [('ai.onnx.ml', 'SVMClassifier'), ('', 'Cast')]
Inputs: [('ingredients', [None, 380], 'tensor(float)')]
Outputs: [('label', [None], 'tensor(string)'), ('probabilities', [None, 5], 'tensor(float)')]


## New recommender assignment

Cuisine classification is the intermediate step. The output is a ranked list of three ingredients to consider adding. For the predicted cuisine, rank absent ingredients by their observed training prevalence; break ties alphabetically. This transparent baseline does not claim to learn personal taste. It excludes ingredients already selected and gives no automatic allergy substitutions.

In [4]:
prevalence = X_train.groupby(y_train).mean().reindex(model.classes_)

def recommend(selected, cuisine, count=3):
    missing = prevalence.loc[cuisine].drop(labels=list(selected))
    return sorted(missing.index, key=lambda name: (-missing[name], name))[:count]

pantry = {"cumin", "cardamom", "yogurt"}
vector = pd.DataFrame([[float(name in pantry) for name in X.columns]], columns=X.columns)
cuisine = model.predict(vector)[0]
suggestions = recommend(pantry, cuisine)
print("Selected:", sorted(pantry))
print("Predicted cuisine:", cuisine)
print("Suggested additions:", suggestions)
assert not pantry.intersection(suggestions)
assert len(suggestions) == len(set(suggestions)) == 3
assert recommend(set(X.columns), cuisine) == []
metadata = {
    "features": list(X.columns), "classes": list(model.classes_),
    "prevalence": prevalence.to_numpy().tolist(),
    "accuracy": float(accuracy_score(y_test, prediction)),
    "macro_f1": float(f1_score(y_test, prediction, average="macro")),
    "source": "Microsoft ML-For-Beginners, 4-Classification/data/cuisines.csv; originally Kaggle Asian and Indian Cuisines",
}
Path("metadata.json").write_text(json.dumps(metadata, indent=2))
fixtures = []
for position in [0, 100, 200]:
    row = X_test.iloc[[position]]
    label = str(model.predict(row)[0])
    selected = list(row.columns[row.iloc[0] == 1])
    fixtures.append({"selected": selected, "label": label, "probabilities": model.predict_proba(row)[0].tolist(), "recommendations": recommend(set(selected), label)})
Path("fixtures.json").write_text(json.dumps(fixtures, indent=2))
print("Saved browser metadata and three cross-runtime test fixtures.")

Selected: ['cardamom', 'cumin', 'yogurt']
Predicted cuisine: indian
Suggested additions: ['turmeric', 'onion', 'coriander']
Saved browser metadata and three cross-runtime test fixtures.


## Model structure and deployment

The printed graph is the structural inspection for this small model: float input [batch, 380], an SVM classification operator, and label/probability outputs. The browser must use the exact exported feature order and class mapping. `predict` labels can differ from the largest calibrated probability for an SVC, so the app uses the label output consistently.

The app loads the model once and builds a fresh binary vector for each request. All model and runtime assets are served locally after npm installation. The source credit is shown on the page. The full UI and cross-runtime checks are described in answers.md; cloud hosting remains excluded.